In [ ]:
!pip install -q streamlit pyjwt bcrypt pyngrok plotly streamlit-option-menu

In [ ]:
%%writefile app.py
import os
import re
import time
import sqlite3
import secrets
import datetime
import smtplib
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

import jwt
import bcrypt
import streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

# --- Securely Fetch Secrets from Colab or Environment ---
try:
    from google.colab import userdata
    JWT_SECRET = userdata.get('JWT_SECRET') or "default-fallback-secret-2026"
    SENDER_EMAIL = userdata.get('EMAIL_ADDRESS') or ""
    EMAIL_PASSWORD = userdata.get('EMAIL_PASSWORD') or ""
except Exception:
    JWT_SECRET = os.getenv("JWT_SECRET", "default-fallback-secret-2026")
    SENDER_EMAIL = os.getenv("EMAIL_ADDRESS", "")
    EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")

OTP_EXPIRY_MINUTES = 5

# --- Configuration & Infosys Corporate Blue Theme Setup ---
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#007cc3"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#007cc3", "accent_hover": "#005b90", "accent_text": "#ffffff",
    "border": "#272343", "border_light": "#bae8e8"
}

# --- CSS Styling ---
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: #ffffff !important; color: #ffffff !important; stroke: #ffffff !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}

    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

# --- Database Helpers ---
def get_db():
    return sqlite3.connect("infosys_portal.db", check_same_thread=False)

def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()

def check_txt(t, h):
    return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE,
        email TEXT UNIQUE,
        password_hash TEXT,
        security_question TEXT,
        security_answer_hash TEXT,
        role TEXT DEFAULT 'user',
        failed_attempts INTEGER DEFAULT 0,
        lock_until TEXT,
        otp_attempts INTEGER DEFAULT 0,
        otp_lock_until TEXT)""")

    # Automatically seed default Admin account if not present (exact match strict)
    admin_exists = conn.execute("SELECT id FROM users WHERE username=?", ("admin",)).fetchone()
    if not admin_exists:
        conn.execute("""INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, role)
                        VALUES (?, ?, ?, ?, ?, ?)""",
                     ("admin", "admin@infosys.com", hash_txt("AdminPassword@123"), "What is your pet's name?", hash_txt("dog"), "admin"))

# --- Input Validation Functions ---
def validate_email(email):
    pattern = r'^[a-zA-Z0-9._%+-]{2,}@[a-zA-Z0-9.-]{2,}\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))

def validate_password(pwd):
    if len(pwd) < 8: return False, "Password must contain at least 8 characters."
    if not re.search(r'[A-Z]', pwd): return False, "Password must contain at least one uppercase letter."
    if not re.search(r'[a-z]', pwd): return False, "Password must contain at least one lowercase letter."
    if not re.search(r'[0-9]', pwd): return False, "Password must contain at least one number."
    if not re.search(r'[@$!%*?&#^()_\-+={}\[\]:;<>,.]', pwd): return False, "Password must contain at least one special symbol."
    return True, "Valid"

# --- JWT Helpers ---
def make_jwt(identity, role="user"):
    payload = {
        "sub": identity,
        "role": role,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None

# --- OTP Helper Functions ---
def generate_otp():
    return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email, otp):
    payload = {
        "sub": email,
        "otp_hash": hash_txt(otp),
        "type": "otp_reset",
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "otp_reset":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]):
            return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"⚠️ OTP expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def send_otp_email(to_email, otp):
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Portal - Password Reset OTP"
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()

    text_body = f"Your OTP for password reset on Infosys Portal is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes."
    html_body = f"""
    <div style="font-family: Arial, sans-serif; padding: 20px; background-color: #f9fcfc; border: 2px solid #272343; border-radius: 12px; max-width: 500px;">
        <h2 style="color: #272343;">Infosys Portal OTP Verification</h2>
        <p style="color: #2d334a;">Use the verification code below to reset your password:</p>
        <div style="background: #007cc3; color: #ffffff; border: 2px solid #272343; padding: 12px 20px; font-size: 28px; font-weight: bold; text-align: center; border-radius: 8px;">{otp}</div>
        <p style="color: #64748b; font-size: 13px; margin-top: 15px;">Valid for {OTP_EXPIRY_MINUTES} minutes.</p>
    </div>
    """
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))

    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(SENDER_EMAIL, EMAIL_PASSWORD)
        server.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        server.quit()
        return True, "OTP sent successfully!"
    except Exception as e:
        return False, f"Failed to send email: {str(e)}"

# --- Session Initialization ---
for k, v in [("token", None), ("page", "Login"), ("forgot_step", "initial"), ("reset_email", None), ("otp_jwt", None), ("login_attempts", 0), ("login_lock_until", None)]:
    if k not in st.session_state:
        st.session_state[k] = v

def navigate(p):
    st.session_state.page = p
    st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING (UNAUTHENTICATED)
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        # --- LOGIN PAGE ---
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            login_id = st.text_input("Username or Email Address", placeholder="username or user@infosys.com").strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Sign In →", use_container_width=True):
                if st.session_state.login_lock_until:
                    if datetime.datetime.utcnow() < st.session_state.login_lock_until:
                        st.error("Account is blocked for 10 minutes. Try again later.")
                        st.stop()
                    else:
                        st.session_state.login_lock_until = None
                        st.session_state.login_attempts = 0

                if not login_id or not pwd:
                    st.session_state.login_attempts += 1
                    if st.session_state.login_attempts >= 5:
                        st.session_state.login_lock_until = datetime.datetime.utcnow() + datetime.timedelta(minutes=10)
                        st.error("Account is blocked for 10 minutes. Try again later.")
                    else:
                        st.error("Enter valid email")
                else:
                    clean_login = login_id.lower().strip()
                    with get_db() as c:
                        r = c.execute("""SELECT email, password_hash, role, failed_attempts, lock_until
                                         FROM users WHERE LOWER(username)=LOWER(?) OR LOWER(email)=LOWER(?)""",
                                      (clean_login, clean_login)).fetchone()

                    if not r:
                        st.session_state.login_attempts += 1
                        if st.session_state.login_attempts >= 5:
                            st.session_state.login_lock_until = datetime.datetime.utcnow() + datetime.timedelta(minutes=10)
                            st.error("Account is blocked for 10 minutes. Try again later.")
                        else:
                            st.error("❌ Invalid credentials.")
                    else:
                        email_db, pwd_hash, role, failed_attempts, lock_until = r

                        if lock_until:
                            lock_time = datetime.datetime.fromisoformat(lock_until)
                            if datetime.datetime.utcnow() < lock_time:
                                st.error("Account is blocked for 10 minutes. Try again later.")
                                st.stop()
                            else:
                                with get_db() as c:
                                    c.execute("UPDATE users SET failed_attempts=0, lock_until=NULL WHERE email=?", (email_db,))
                                failed_attempts = 0

                        if check_txt(pwd, pwd_hash):
                            st.session_state.login_attempts = 0
                            st.session_state.login_lock_until = None
                            with get_db() as c:
                                c.execute("UPDATE users SET failed_attempts=0, lock_until=NULL WHERE email=?", (email_db,))
                            st.session_state.token = make_jwt(email_db, role=role)
                            navigate("Dashboard")
                        else:
                            failed_attempts += 1
                            st.session_state.login_attempts += 1
                            if failed_attempts >= 5 or st.session_state.login_attempts >= 5:
                                new_lock = (datetime.datetime.utcnow() + datetime.timedelta(minutes=10)).isoformat()
                                with get_db() as c:
                                    c.execute("UPDATE users SET failed_attempts=?, lock_until=? WHERE email=?", (failed_attempts, new_lock, email_db))
                                st.session_state.login_lock_until = datetime.datetime.utcnow() + datetime.timedelta(minutes=10)
                                st.error("Account is blocked for 10 minutes. Try again later.")
                            else:
                                with get_db() as c:
                                    c.execute("UPDATE users SET failed_attempts=? WHERE email=?", (failed_attempts, email_db))
                                st.error("❌ Invalid credentials.")

            st.markdown("---")
            col_a, col_b = st.columns(2)
            if col_a.button("Create Account", use_container_width=True): navigate("Signup")
            if col_b.button("Forgot Password?", use_container_width=True):
                st.session_state.forgot_step = "initial"
                navigate("Forgot")

        # --- SIGNUP PAGE ---
        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Username", placeholder="e.g. janesmith").strip()
            email = st.text_input("Email address", placeholder="e.g. jane@company.com").strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm Password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", [
                "What is your pet's name?",
                "What is your mother's maiden name?",
                "What was your first school name?"
            ])
            sa = st.text_input("Security Answer", placeholder="Your secret answer").strip() # Strict exact case preserved
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account →", use_container_width=True):
                valid_pw, pw_msg = validate_password(pwd)
                clean_email = email.lower().strip()

                if not uname:
                    st.error("Enter valid username")
                elif not clean_email or not validate_email(clean_email):
                    st.error("Enter valid email")
                elif not pwd:
                    st.error("Enter valid password")
                elif not valid_pw:
                    st.error(pw_msg)
                elif not confirm_pwd:
                    st.error("Confirm password is required")
                elif pwd != confirm_pwd:
                    st.error("Passwords do not match")
                elif not sa:
                    st.error("Enter valid security answer")
                else:
                    try:
                        with get_db() as c:
                            c.execute("""INSERT INTO users (username, email, password_hash, security_question, security_answer_hash, role)
                                         VALUES (?, ?, ?, ?, ?, ?)""",
                                      (uname, clean_email, hash_txt(pwd), sq, hash_txt(sa), 'user')) # Exact case stored
                        st.success("✅ Account created successfully! Please log in.")
                        time.sleep(1.5)
                        navigate("Login")
                    except sqlite3.IntegrityError:
                        st.error("❌ Username or Email is already registered. Please choose another.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        # --- FORGOT PASSWORD PAGE ---
        elif st.session_state.page == "Forgot":
            auth_header("Reset Password", "Recover access to your account")

            if st.session_state.forgot_step == "initial":
                method = st.radio("Choose Recovery Method:", ["Security Question Route", "OTP Email Route"])
                user_id = st.text_input("Username or Registered Email", placeholder="Enter username or email").strip()
                st.markdown("<br>", unsafe_allow_html=True)

                if st.button("Proceed →", use_container_width=True):
                    if not user_id:
                        st.error("⚠️ Please enter your username or registered email.")
                    else:
                        clean_uid = user_id.lower().strip()
                        with get_db() as c:
                            r = c.execute("""SELECT email, security_question, security_answer_hash, otp_attempts, otp_lock_until
                                             FROM users WHERE LOWER(username)=LOWER(?) OR LOWER(email)=LOWER(?)""",
                                          (clean_uid, clean_uid)).fetchone()
                        if not r:
                            st.error("❌ Account not found.")
                        else:
                            st.session_state.reset_email = r[0]
                            st.session_state.sq_q = r[1]
                            st.session_state.sq_a_hash = r[2]

                            lock_dt = r[4]
                            if lock_dt and datetime.datetime.utcnow() < datetime.datetime.fromisoformat(lock_dt):
                                st.error("Account is blocked for 10 minutes. Try again later.")
                                st.stop()

                            if method == "Security Question Route":
                                st.session_state.forgot_step = "sq_verify"
                                st.rerun()
                            else:
                                if not SENDER_EMAIL or not EMAIL_PASSWORD:
                                    st.error("❌ Gmail secrets are missing in configuration.")
                                else:
                                    otp = generate_otp()
                                    ok, msg = send_otp_email(st.session_state.reset_email, otp)
                                    if ok:
                                        st.session_state.otp_jwt = make_otp_token(st.session_state.reset_email, otp)
                                        st.session_state.forgot_step = "otp_verify"
                                        st.success("✅ OTP sent to your email!")
                                        time.sleep(1)
                                        st.rerun()
                                    else:
                                        st.error(f"❌ {msg}")

            elif st.session_state.forgot_step == "sq_verify":
                st.info(f"❓ **Security Question:** {st.session_state.sq_q}")
                ans = st.text_input("Your Answer", placeholder="Enter your answer").strip() # Strict exact case
                npw = st.text_input("New Password", type="password")
                confirm_npw = st.text_input("Confirm New Password", type="password")
                st.markdown("<br>", unsafe_allow_html=True)

                if st.button("Reset Password →", use_container_width=True):
                    with get_db() as c:
                        user_rec = c.execute("SELECT otp_attempts, otp_lock_until FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                    otp_attempts, otp_lock_until = user_rec[0], user_rec[1]

                    if otp_lock_until and datetime.datetime.utcnow() < datetime.datetime.fromisoformat(otp_lock_until):
                        st.error("Account is blocked for 10 minutes. Try again later.")
                        st.stop()

                    valid_pw, pw_msg = validate_password(npw)
                    if not ans:
                        st.error("Enter valid security answer")
                    elif not npw:
                        st.error("Enter valid password")
                    elif not check_txt(ans, st.session_state.sq_a_hash): # Strict exact case match check against stored hash
                        otp_attempts += 1
                        if otp_attempts >= 5:
                            new_lock = (datetime.datetime.utcnow() + datetime.timedelta(minutes=10)).isoformat()
                            with get_db() as c:
                                c.execute("UPDATE users SET otp_attempts=?, otp_lock_until=? WHERE email=?", (otp_attempts, new_lock, st.session_state.reset_email))
                            st.error("Account is blocked for 10 minutes. Try again later.")
                        else:
                            with get_db() as c:
                                c.execute("UPDATE users SET otp_attempts=? WHERE email=?", (otp_attempts, st.session_state.reset_email))
                            st.error(f"❌ Incorrect security answer. ({5 - otp_attempts} attempts remaining)")
                    elif not valid_pw:
                        st.error(pw_msg)
                    elif npw != confirm_npw:
                        st.error("Passwords do not match")
                    else:
                        with get_db() as c:
                            c.execute("UPDATE users SET password_hash=?, otp_attempts=0, otp_lock_until=NULL WHERE email=?",
                                      (hash_txt(npw), st.session_state.reset_email))
                        st.success("🎉 Password updated successfully!")
                        time.sleep(1.5)
                        navigate("Login")

            elif st.session_state.forgot_step == "otp_verify":
                st.info(f"📧 An OTP has been sent to **{st.session_state.reset_email}** (Expires in 5 mins).")
                otp_in = st.text_input("6-Digit OTP Code", max_chars=6).strip()
                npw = st.text_input("New Password", type="password")
                confirm_npw = st.text_input("Confirm New Password", type="password")
                st.markdown("<br>", unsafe_allow_html=True)

                if st.button("Verify OTP & Reset Password →", use_container_width=True):
                    with get_db() as c:
                        user_rec = c.execute("SELECT otp_attempts, otp_lock_until FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                    otp_attempts, otp_lock_until = user_rec[0], user_rec[1]

                    if otp_lock_until and datetime.datetime.utcnow() < datetime.datetime.fromisoformat(otp_lock_until):
                        st.error("Account is blocked for 10 minutes. Try again later.")
                        st.stop()

                    valid_pw, pw_msg = validate_password(npw)
                    if not otp_in:
                        st.error("Enter valid OTP code")
                    elif not npw:
                        st.error("Enter valid password")
                    else:
                        ok, msg = verify_otp_token(st.session_state.otp_jwt, otp_in, st.session_state.reset_email)
                        if not ok:
                            otp_attempts += 1
                            if otp_attempts >= 5:
                                new_lock = (datetime.datetime.utcnow() + datetime.timedelta(minutes=10)).isoformat()
                                with get_db() as c:
                                    c.execute("UPDATE users SET otp_attempts=?, otp_lock_until=? WHERE email=?", (otp_attempts, new_lock, st.session_state.reset_email))
                                st.error("Account is blocked for 10 minutes. Try again later.")
                            else:
                                with get_db() as c:
                                    c.execute("UPDATE users SET otp_attempts=? WHERE email=?", (otp_attempts, st.session_state.reset_email))
                                st.error(f"❌ {msg} ({5 - otp_attempts} attempts remaining)")
                        elif not valid_pw:
                            st.error(pw_msg)
                        elif npw != confirm_npw:
                            st.error("Passwords do not match")
                        else:
                            with get_db() as c:
                                c.execute("UPDATE users SET password_hash=?, otp_attempts=0, otp_lock_until=NULL WHERE email=?",
                                          (hash_txt(npw), st.session_state.reset_email))
                            st.success("🎉 Password updated successfully!")
                            time.sleep(1.5)
                            navigate("Login")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.forgot_step = "initial"
                navigate("Login")

# ============================================================
# PAGE ROUTING (AUTHENTICATED SESSION)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    identity = payload["sub"]
    role = payload.get("role", "user")

    # --- Sidebar Navigation ---
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Dashboard" if role == "admin" else "User Dashboard"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Logout"]
        menu = option_menu(None, opts, icons=["house", "box-arrow-right"],
                            styles={"container": {"background-color": COLORS['bg_sidebar']},
                                    "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # --- ADMIN DASHBOARD ---
    if role == "admin":
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">🛡️ Admin Dashboard</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Registered User Management</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['accent_text']};">Logged in as: {identity}</div>
        </div>
        """, unsafe_allow_html=True)

        st.subheader("Registered Users Overview")
        with get_db() as c:
            users_data = c.execute("SELECT id, username, email, role FROM users").fetchall()

        if users_data:
            st.table([{"User ID": u[0], "Username": u[1], "Email": u[2], "Role": u[3]} for u in users_data])
        else:
            st.info("No registered users found in database.")

    # --- USER DASHBOARD ---
    else:
        with get_db() as c:
            r = c.execute("SELECT username FROM users WHERE email=?", (identity,)).fetchone()
            uname = r[0] if r else identity

        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Welcome, {uname}!</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Infosys Analytics Workspace</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['accent_text']};">👤 {identity}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(
            mode="gauge+number",
            value=95,
            title={"text": "System Performance Index", "font": {"color": COLORS['text_heading'], "size": 16}},
            gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}
        ))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=280, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# Retrieve secrets safely from Colab Secrets
try:
    os.environ['JWT_SECRET'] = userdata.get('JWT_SECRET')
    os.environ['NGROK_AUTHTOKEN'] = userdata.get('NGROK_AUTHTOKEN')
    os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
    os.environ['EMAIL_ADDRESS'] = userdata.get('EMAIL_ADDRESS')

    ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
except Exception as e:
    print("⚠️ Error retrieving Colab secrets! Ensure JWT_SECRET, NGROK_AUTHTOKEN, EMAIL_PASSWORD, and EMAIL_ADDRESS are configured in Colab Secrets.")

# Clean up old running sessions
!pkill -f streamlit
ngrok.kill()

# Start Streamlit application
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

# Connect ngrok tunnel
try:
    public_url = ngrok.connect(8501).public_url
    print("=" * 65)
    print(f"🚀 Infosys Springboard Milestone 1 URL: {public_url}")
    print("=" * 65)
    print("⏳ Application is live! Press [Ctrl + C] or stop the cell to shut down.")

    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down server...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped cleanly.")